# FIGNN — Research Validation & Robustness Notebook

This notebook is for **research validation**, not presentation.

It consolidates the completed Stage-1/Stage-2 analyses and adds the targeted checks motivated by the review corrections:

1. Audit the actual training/validation split.
2. Preserve silhouette as a **development diagnostic**, not the primary transferability metric.
3. Recheck the completed 29-subject trajectory geometry.
4. Test the 3-point Procrustes result with **1,000 random workload-label alterations per subject**.
5. Examine the REST anchor using four-condition latent-space visualization and REST-to-N-back geometry.
6. Consolidate Stage-2 error statistics and paired inference.
7. Search for historical smaller-architecture checkpoints before deciding whether a new 8-head ablation must be trained.

**No model training is performed by default.** Frozen-encoder embedding extraction is performed only for the new REST/Procrustes analyses.


In [1]:
# ============================================================
# 0. SETUP
# ============================================================

import os
import sys
import re
import glob
import json
import copy
import pickle
import warnings

import numpy as np
import pandas as pd
import torch
import plotly.graph_objects as go
import plotly.express as px

from scipy.spatial import procrustes
from scipy.stats import wilcoxon, ttest_rel
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
from torch_geometric.loader import DataLoader

warnings.filterwarnings("ignore")

PROJECT_ROOT = "/home/nvidia/24BCE1793/fignn_cni__project"
GRAPH_PATH = "/lp-dev/24BCE1793/data/processed/all_graphs_29sub.pkl"
LOSO_ROOT = "/lp-dev/24BCE1793/outputs/loso"
STAGE1_ROOT = "/lp-dev/24BCE1793/outputs/evaluation_loso_all29"
STAGE2_CAL_PATH = "/lp-dev/24BCE1793/outputs/stage2_calibration_comparison_64D.csv"
STAGE2_DIST_PATH = "/lp-dev/24BCE1793/outputs/stage2_centroid_64D_distances.csv"

SUBJECTS = list(range(1, 30))
TASK_LABELS = [0, 1, 2]
REST_LABEL = 3
LABEL_NAMES = {0: "0-back", 1: "1-back", 2: "2-back", 3: "REST"}

N_PERM = 1000
RNG_SEED = 20260906

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

sys.path.insert(0, PROJECT_ROOT)
from src.models.model import FIGNNEncoder

print("Project:", PROJECT_ROOT)
print("Device:", DEVICE)
print("Permutation draws per subject:", N_PERM)


/lp-dev/24BCE1793/fignn_env/lib/python3.13/site-packages/torch/cuda/__init__.py:64: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
/lp-dev/24BCE1793/fignn_env/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Project: /home/nvidia/24BCE1793/fignn_cni__project
Device: cuda
Permutation draws per subject: 1000


In [2]:
# ============================================================
# 1. HELPERS
# ============================================================

def get_subject(g):
    for name in ["subject", "subj", "subject_id", "sid"]:
        if hasattr(g, name):
            value = getattr(g, name)
            return int(value.item()) if torch.is_tensor(value) else int(value)
    raise AttributeError("Could not find subject ID.")

def get_label(g):
    return int(g.y.item())

def compute_train_stats(graphs):
    all_x = torch.cat([g.x for g in graphs], dim=0)
    mean = all_x.mean(dim=0)
    std = all_x.std(dim=0).clamp(min=1e-8)
    return mean, std

def normalize_graphs(graphs, mean, std):
    out = []
    for g in graphs:
        h = copy.copy(g)
        h.x = (g.x.clone() - mean) / std
        out.append(h)
    return out

@torch.no_grad()
def encode_graphs(model, graphs, mean, std, batch_size=128):
    if not graphs:
        return np.empty((0, 64)), np.empty((0,), dtype=int)
    loader = DataLoader(
        normalize_graphs(graphs, mean, std),
        batch_size=batch_size,
        shuffle=False,
        num_workers=0
    )
    zs, ys = [], []
    model.eval()
    for batch in loader:
        batch = batch.to(DEVICE)
        z = model(batch)
        zs.append(z.cpu().numpy())
        ys.append(batch.y.view(-1).cpu().numpy())
    return np.vstack(zs), np.concatenate(ys)

def centroid(z, y, label):
    return z[y == label].mean(axis=0)

def trajectory(z, y):
    return np.vstack([centroid(z, y, l) for l in TASK_LABELS])

def trajectory_geometry(T):
    d01 = np.linalg.norm(T[1] - T[0])
    d12 = np.linalg.norm(T[2] - T[1])
    d02 = np.linalg.norm(T[2] - T[0])
    a = T[0] - T[1]
    b = T[2] - T[1]
    denom = np.linalg.norm(a) * np.linalg.norm(b)
    angle = (
        np.degrees(np.arccos(np.clip(np.dot(a, b) / denom, -1, 1)))
        if denom > 1e-12 else np.nan
    )
    efficiency = d02 / (d01 + d12) if (d01 + d12) > 1e-12 else np.nan
    return d01, d12, d02, angle, efficiency

def load_fold_model(subject, checkpoint_name="fignn_best.pth"):
    ckpt = os.path.join(LOSO_ROOT, f"subject_{subject:02d}", checkpoint_name)
    if not os.path.exists(ckpt):
        raise FileNotFoundError(ckpt)
    model = FIGNNEncoder().to(DEVICE)
    model.load_state_dict(torch.load(ckpt, map_location=DEVICE))
    model.eval()
    return model

def load_fold_stats(subject, training_graphs):
    path = os.path.join(
        LOSO_ROOT, f"subject_{subject:02d}", "normalisation_stats.pt"
    )
    if os.path.exists(path):
        s = torch.load(path, map_location="cpu")
        return s["mean"].float(), s["std"].float().clamp(min=1e-8)
    return compute_train_stats(training_graphs)

def bh_fdr(pvals):
    pvals = np.asarray(pvals, dtype=float)
    order = np.argsort(pvals)
    ranked = pvals[order]
    q = ranked * len(pvals) / np.arange(1, len(pvals) + 1)
    q = np.minimum.accumulate(q[::-1])[::-1]
    out = np.empty_like(q)
    out[order] = q
    return np.minimum(out, 1.0)


## 2. Training / validation protocol audit

The final LOSO training protocol uses:

- strict subject-wise LOSO;
- a subject-wise validation split among the 28 training subjects;
- training-only normalization;
- balanced graph sampling;
- validation-loss checkpoint selection.

The held-out LOSO subject is excluded from both training and validation.

This section verifies that the implementation matches the final protocol used to generate the authoritative Stage-1 checkpoints.

In [3]:
# ============================================================
# 2. TRAINING / VALIDATION PROTOCOL AUDIT
# ============================================================

from pathlib import Path

TRAIN_SCRIPT = Path(
    "/home/nvidia/24BCE1793/fignn_cni__project/src/training/train.py"
)

LOSO_SCRIPT = Path(
    PROJECT_ROOT + "/loso.py"
)

train_text = TRAIN_SCRIPT.read_text()
loso_text = LOSO_SCRIPT.read_text()

checks = {
    "LOSO runner excludes held-out subject": (
        "test_subject" in loso_text
        and "train_graphs" in loso_text
    ),
    "Subject-wise validation split": (
        "make_subjectwise_split" in train_text
        and "val_split" in train_text
    ),
    "Training-only normalization": (
        "normalisation_stats" in train_text
        or "normalization_stats" in train_text
    ),
    "Balanced graph sampling": (
        "WeightedRandomSampler" in train_text
    ),
    "Validation-loss checkpoint selection": (
        "best_val_loss" in train_text
    ),
}

for name, passed in checks.items():
    print(f"{'PASS' if passed else 'CHECK'}: {name}")
# ============================================================
# VERIFY FINAL LOSO FOLD ARTIFACTS
# ============================================================

for subject in SUBJECTS:
    fold_dir = Path(LOSO_ROOT) / f"subject_{subject:02d}"

    assert (fold_dir / "fignn_best.pth").exists()
    assert (fold_dir / "normalisation_stats.pt").exists()

print(
    f"PASS: {len(SUBJECTS)}/29 LOSO folds contain "
    "best checkpoints and normalization statistics."
)

CHECK: LOSO runner excludes held-out subject
PASS: Subject-wise validation split
PASS: Training-only normalization
PASS: Balanced graph sampling
PASS: Validation-loss checkpoint selection
PASS: 29/29 LOSO folds contain best checkpoints and normalization statistics.


## 3. Silhouette: retain as a POST-HOC metric

Silhouette remains in the research record, but its role is explicit: Silhouette is retained as a post-hoc descriptive development diagnostic. It is not used for final checkpoint selection and is not treated as the primary transferability metric.

In [4]:
# ============================================================
# POST-HOC LOSO SILHOUETTE ANALYSIS
# ============================================================

with open(GRAPH_PATH, "rb") as f:
    all_graphs = pickle.load(f)

print(f"Loaded {len(all_graphs)} graphs")

sil_rows = []

for subject in range(1, 30):
    ckpt_path = os.path.join(LOSO_ROOT, f"subject_{subject:02d}", "fignn_best.pth")
    stats_path = os.path.join(LOSO_ROOT, f"subject_{subject:02d}", "normalisation_stats.pt")

    if not os.path.exists(ckpt_path) or not os.path.exists(stats_path):
        print(f"Subject {subject:02d}: missing checkpoint/statistics")
        continue

    stats = torch.load(stats_path, map_location="cpu", weights_only=True)
    mean, std = stats["mean"].float(), stats["std"].float().clamp(min=1e-8)

    test_graphs = [g for g in all_graphs if int(g.subject) == subject]
    test_graphs = [g.clone() for g in test_graphs]

    for g in test_graphs:
        g.x = (g.x - mean) / std

    model = FIGNNEncoder(node_in_dim=4, edge_dim=4, hidden_dim=128, latent_dim=64, n_heads=8, dropout=0.3).to(DEVICE)
    model.load_state_dict(torch.load(ckpt_path, map_location=DEVICE, weights_only=True))
    model.eval()

    embeddings, labels = [], []

    with torch.no_grad():
        for g in test_graphs:
            g = g.to(DEVICE)
            z = model(g)
            embeddings.append(torch.nn.functional.normalize(z, p=2, dim=1).cpu().numpy())
            labels.extend(g.y.cpu().numpy().tolist())

    embeddings = np.concatenate(embeddings, axis=0)
    labels = np.asarray(labels)

    if len(np.unique(labels)) < 2:
        print(f"Subject {subject:02d}: insufficient classes")
        continue

    sil_rows.append({"subject": subject, "silhouette": silhouette_score(embeddings, labels, metric="euclidean")})
    print(f"Subject {subject:02d}: silhouette = {sil_rows[-1]['silhouette']:.4f}")

silhouette_df = pd.DataFrame(sil_rows).sort_values("subject").reset_index(drop=True)
display(silhouette_df.round(4))

print(f"\nMean silhouette: {silhouette_df['silhouette'].mean():.4f} ± {silhouette_df['silhouette'].std():.4f}")
print(f"Median silhouette: {silhouette_df['silhouette'].median():.4f}")
print(f"Positive silhouette: {(silhouette_df['silhouette'] > 0).sum()}/{len(silhouette_df)}")
silhouette_df.to_csv(os.path.join(LOSO_ROOT, "loso_all29_silhouette.csv"), index=False)

Loaded 27666 graphs
Subject 01: silhouette = -0.0972
Subject 02: silhouette = 0.0199
Subject 03: silhouette = -0.0223
Subject 04: silhouette = -0.0074
Subject 05: silhouette = -0.0260
Subject 06: silhouette = -0.0677
Subject 07: silhouette = -0.0021
Subject 08: silhouette = -0.0404
Subject 09: silhouette = 0.0422
Subject 10: silhouette = -0.0354
Subject 11: silhouette = -0.1415
Subject 12: silhouette = -0.0142
Subject 13: silhouette = -0.0209
Subject 14: silhouette = -0.0164
Subject 15: silhouette = -0.0254
Subject 16: silhouette = -0.0301
Subject 17: silhouette = -0.0177
Subject 18: silhouette = 0.0122
Subject 19: silhouette = 0.0276
Subject 20: silhouette = -0.0499
Subject 21: silhouette = -0.0169
Subject 22: silhouette = 0.0150
Subject 23: silhouette = -0.0266
Subject 24: silhouette = -0.0552
Subject 25: silhouette = -0.0665
Subject 26: silhouette = -0.0898
Subject 27: silhouette = -0.0280
Subject 28: silhouette = -0.0492
Subject 29: silhouette = -0.1257


,subject,silhouette
0,1,-0.0972
1,2,0.0199
2,3,-0.0223
3,4,-0.0074
4,5,-0.0260
5,6,-0.0677
6,7,-0.0021
7,8,-0.0404
8,9,0.0422
9,10,-0.0354



Mean silhouette: -0.0329 ± 0.0426
Median silhouette: -0.0260
Positive silhouette: 5/29


## 4. Completed Stage-1 geometry — verification only

The distance/trajectory analysis is **already completed**. This section loads the authoritative all-29 LOSO results and does not introduce a new monotonicity test.

Existing analyses already computed \(d_{01}\), \(d_{12}\), \(d_{02}\), trajectory angle, path efficiency, and the 28/29 ordering result. 

In [16]:
# ============================================================
# 4. AUTHORITATIVE STAGE-1 RESULTS
# ============================================================

stage1_csv=os.path.join(STAGE1_ROOT,"loso_all29_subject_metrics.csv")
stage1_df=pd.read_csv(stage1_csv).sort_values("subject").reset_index(drop=True)

print("Subjects:",len(stage1_df))
display(stage1_df.round(4))

print(f"\nMean Procrustes similarity: {stage1_df['procrustes_similarity'].mean():.4f}")
print(f"d01/d02 < 1: {(stage1_df['d01_d02']<1).sum()}/{len(stage1_df)}")
print(f"1B between 0B and 2B: {stage1_df['oneback_between'].sum()}/{len(stage1_df)}")

Subjects: 29


,subject,d01,d02,d12,d01_d02,d12_d02,angle_deg,path_efficiency,relative_1back_position,perpendicular_distance,centrality,oneback_between,procrustes_similarity,direction_cosine,n_train_trajectories,checkpoint
0,1,0.0121,0.0217,0.0149,0.5592,0.6888,73.9578,0.8013,0.4191,0.0080,0.8383,True,0.9034,-0.8348,28,/lp-dev/24BCE1793/outputs/loso/subject_01/fign...
1,2,0.0247,0.0885,0.0776,0.2790,0.8771,71.8142,0.8649,0.1542,0.0206,0.3085,True,0.9629,0.8755,28,/lp-dev/24BCE1793/outputs/loso/subject_02/fign...
2,3,0.0173,0.0442,0.0336,0.3920,0.7591,63.0212,0.8687,0.2887,0.0117,0.5774,True,0.9654,0.8543,28,/lp-dev/24BCE1793/outputs/loso/subject_03/fign...
3,4,0.0434,0.0674,0.0282,0.6450,0.4187,40.8078,0.9402,0.6204,0.0119,0.7593,True,0.8095,0.9548,28,/lp-dev/24BCE1793/outputs/loso/subject_04/fign...
4,5,0.0322,0.0599,0.0339,0.5372,0.5656,49.8851,0.9068,0.4843,0.0139,0.9686,True,0.9127,0.9430,28,/lp-dev/24BCE1793/outputs/loso/subject_05/fign...
5,6,0.0325,0.0597,0.0357,0.5439,0.5980,57.8163,0.8757,0.4691,0.0164,0.9383,True,0.9079,0.9520,28,/lp-dev/24BCE1793/outputs/loso/subject_06/fign...
6,7,0.0118,0.0360,0.0257,0.3282,0.7139,35.2401,0.9596,0.2991,0.0049,0.5981,True,0.9984,0.8621,28,/lp-dev/24BCE1793/outputs/loso/subject_07/fign...
7,8,0.0202,0.0455,0.0267,0.4450,0.5865,28.6813,0.9694,0.4270,0.0057,0.8540,True,0.9645,0.7657,28,/lp-dev/24BCE1793/outputs/loso/subject_08/fign...
8,9,0.0108,0.0645,0.0652,0.1678,1.0110,98.5272,0.8483,0.0030,0.0108,0.0060,True,0.9440,0.9650,28,/lp-dev/24BCE1793/outputs/loso/subject_09/fign...
9,10,0.0129,0.0695,0.0619,0.1857,0.8913,58.8951,0.9285,0.1200,0.0099,0.2400,True,0.9741,0.9458,28,/lp-dev/24BCE1793/outputs/loso/subject_10/fign...



Mean Procrustes similarity: 0.8775
d01/d02 < 1: 27/29
1B between 0B and 2B: 23/29


## 5. Procrustes permutation null — primary new robustness test

For every held-out subject:

1. keep the subject's three latent centroids unchanged;
2. randomly permute their workload labels;
3. compute Procrustes similarity against the fold-specific population trajectory;
4. repeat **1,000 times**;
5. compare the observed similarity with the null distribution.

The correction specifically recommends 1,000 random label permutations per subject to test whether the observed similarity is achievable by chance. 

The notebook uses the fold-specific `fignn_best.pth` checkpoint and training-only normalization statistics so the null is evaluated in the same frozen LOSO setting as the structured trajectory analysis.

In [17]:
# ============================================================
# 5. PROCRUSTES PERMUTATION NULL
# ============================================================

all_graphs = None
trajectory_cache = {}
population_cache = {}

def load_graphs_once():
    global all_graphs
    if all_graphs is None:
        print("Loading graphs...")
        with open(GRAPH_PATH, "rb") as f:
            all_graphs = pickle.load(f)
        print("Graphs:", len(all_graphs))
    return all_graphs

def get_fold_trajectories(subject):
    """Return population and held-out 3x64 trajectories for one LOSO fold."""
    if subject in trajectory_cache:
        return population_cache[subject], trajectory_cache[subject]

    graphs = load_graphs_once()

    train_graphs = [
        g for g in graphs
        if get_subject(g) != subject and get_label(g) in TASK_LABELS
    ]
    test_graphs = [
        g for g in graphs
        if get_subject(g) == subject and get_label(g) in TASK_LABELS
    ]

    model = load_fold_model(subject, "fignn_best.pth")
    mean, std = load_fold_stats(subject, train_graphs)

    z_train, y_train = encode_graphs(model, train_graphs, mean, std)
    z_test, y_test = encode_graphs(model, test_graphs, mean, std)

    training_subjects = sorted(set(get_subject(g) for g in train_graphs))

    pop = []
    for label in TASK_LABELS:
        per_subject = []
        for sid in training_subjects:
            mask = np.array([
                get_subject(g) == sid and get_label(g) == label
                for g in train_graphs
            ])
            per_subject.append(z_train[mask].mean(axis=0))
        pop.append(np.mean(per_subject, axis=0))

    pop = np.vstack(pop)
    subj = trajectory(z_test, y_test)

    population_cache[subject] = pop
    trajectory_cache[subject] = subj

    del model, z_train, z_test
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return pop, subj

rng = np.random.default_rng(RNG_SEED)
perm_rows = []

for _, row in stage1_df.sort_values("subject").iterrows():
    sid = int(row["subject"])

    pop_T, subj_T = get_fold_trajectories(sid)

    observed_sim = 1.0 - procrustes(pop_T, subj_T)[2]

    null_sims = np.empty(N_PERM)
    for j in range(N_PERM):
        perm = rng.permutation(3)
        null_sims[j] = 1.0 - procrustes(pop_T, subj_T[perm])[2]

    ge = int(np.sum(null_sims >= observed_sim))
    p_raw = ge / N_PERM
    p_plus1 = (ge + 1) / (N_PERM + 1)
    percentile = 100.0 * np.mean(null_sims < observed_sim)

    perm_rows.append({
        "subject": sid,
        "observed_procrustes": observed_sim,
        "null_mean": null_sims.mean(),
        "null_sd": null_sims.std(ddof=1),
        "null_q025": np.quantile(null_sims, 0.025),
        "null_q975": np.quantile(null_sims, 0.975),
        "null_ge_observed": ge,
        "empirical_p_raw": p_raw,
        "empirical_p_plus1": p_plus1,
        "observed_percentile": percentile,
    })

perm_df = pd.DataFrame(perm_rows).sort_values("subject").reset_index(drop=True)
perm_df["q_fdr"] = bh_fdr(perm_df["empirical_p_plus1"].values)

display(perm_df.round(5))

print("\nSubjects with p < 0.05:",
      int((perm_df["empirical_p_plus1"] < 0.05).sum()), "/", len(perm_df))
print("Subjects with FDR q < 0.05:",
      int((perm_df["q_fdr"] < 0.05).sum()), "/", len(perm_df))
print("Median empirical p:",
      float(perm_df["empirical_p_plus1"].median()))
print("Median observed percentile:",
      float(perm_df["observed_percentile"].median()))


Loading graphs...
Graphs: 27666


,subject,observed_procrustes,null_mean,null_sd,null_q025,null_q975,null_ge_observed,empirical_p_raw,empirical_p_plus1,observed_percentile,q_fdr
0,1,0.90342,0.56867,0.23634,0.25763,0.90342,144,0.144,0.14486,85.6,0.32039
1,2,0.96286,0.53973,0.31204,0.11655,0.96286,162,0.162,0.16284,83.8,0.32039
2,3,0.96542,0.53322,0.29410,0.13283,0.96542,146,0.146,0.14685,85.4,0.32039
3,4,0.80954,0.56281,0.30990,0.09478,0.96374,333,0.333,0.33367,66.7,0.41487
4,5,0.91273,0.53496,0.29435,0.15871,0.91273,153,0.153,0.15385,84.7,0.32039
5,6,0.90793,0.56416,0.28269,0.19529,0.90793,165,0.165,0.16583,83.5,0.32039
6,7,0.99841,0.53421,0.33171,0.06620,0.99841,166,0.166,0.16683,83.4,0.32039
7,8,0.96452,0.54387,0.33627,0.08423,0.96452,185,0.185,0.18581,81.5,0.32039
8,9,0.94396,0.52223,0.32872,0.13641,0.95224,319,0.319,0.31968,68.1,0.41487
9,10,0.97413,0.50841,0.32911,0.07292,0.97413,159,0.159,0.15984,84.1,0.32039



Subjects with p < 0.05: 0 / 29
Subjects with FDR q < 0.05: 0 / 29
Median empirical p: 0.17382617382617382
Median observed percentile: 82.69999999999999


In [18]:
# ============================================================
# 5A. PROCRUSTES NULL SUMMARY PLOT
# ============================================================

fig = px.scatter(
    perm_df,
    x="subject",
    y="observed_percentile",
    hover_data=["empirical_p_plus1", "observed_procrustes"],
    title="Observed Procrustes Similarity: Percentile Within 1,000-Draw Null"
)

fig.add_hline(y=95, line_dash="dash")
fig.update_layout(
    template="plotly_white",
    xaxis_title="Held-out subject",
    yaxis_title="Observed percentile in permutation null",
    height=500
)
fig.show()


## 6. Procrustes bootstrap stability

This analysis assesses the stability of the subject-level Procrustes similarity estimates under resampling of held-out epochs.

For each held-out subject, epochs within each N-back workload condition are resampled with replacement 1,000 times. The three workload centroids are recomputed for each bootstrap sample, and Procrustes similarity is evaluated against the corresponding population trajectory.

The bootstrap analysis reports the subject-level observed Procrustes similarity together with its 95% bootstrap confidence interval. The observed values are computed using the same fold-specific embeddings and trajectory definition as the authoritative Stage-1 analysis.

This analysis is used as a **stability assessment**, rather than as a statistical significance test of the Procrustes metric.

In [19]:
# ============================================================
# 6. PROCRUSTES BOOTSTRAP STABILITY — CORRECTED
# ============================================================
#
# Purpose:
#   Quantify uncertainty in the held-out subject trajectory
#   while keeping the authoritative Stage-1 CSV Procrustes
#   result as the observed reference.
#
# IMPORTANT:
#   - 1,000 bootstrap resamples per subject
#   - resample held-out epochs within each workload condition
#   - recompute the 3 workload centroids for each bootstrap
#   - compare each bootstrap trajectory against the SAME
#     fold-specific population trajectory
#   - observed Procrustes is taken directly from the
#     authoritative Stage-1 CSV.
#
# This avoids replacing the authoritative Stage-1 calculation
# with a separately reconstructed observed statistic.
# ============================================================

N_BOOT = 1000
bootstrap_rng = np.random.default_rng(RNG_SEED)

# ------------------------------------------------------------
# Cache epoch-level held-out latent embeddings
# ------------------------------------------------------------

embedding_cache = {}

def get_fold_test_embeddings_for_bootstrap(subject):
    if subject in embedding_cache:
        return embedding_cache[subject]

    graphs = load_graphs_once()

    train_graphs = [
        g for g in graphs
        if get_subject(g) != subject
        and get_label(g) in TASK_LABELS
    ]

    test_graphs = [
        g for g in graphs
        if get_subject(g) == subject
        and get_label(g) in TASK_LABELS
    ]

    model = load_fold_model(subject, "fignn_best.pth")
    mean, std = load_fold_stats(subject, train_graphs)

    z_test, y_test = encode_graphs(
        model,
        test_graphs,
        mean,
        std
    )

    embedding_cache[subject] = (z_test, y_test)

    del model

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return embedding_cache[subject]


# ------------------------------------------------------------
# Bootstrap
# ------------------------------------------------------------

bootstrap_rows = []

for _, row in stage1_df.sort_values("subject").iterrows():

    sid = int(row["subject"])

    # Fold-specific population trajectory
    pop_T, _ = get_fold_trajectories(sid)

    # Authoritative Stage-1 Procrustes from current CSV
    possible_cols = [
        "procrustes_similarity",
        "procrustes",
        "procrustes_sim",
    ]

    observed_col = next(
        (c for c in possible_cols if c in stage1_df.columns),
        None
    )

    if observed_col is None:
        raise KeyError(
            "Could not find authoritative Procrustes column "
            f"in stage1_df. Available columns: {list(stage1_df.columns)}"
        )

    observed_sim = float(row[observed_col])

    # Held-out epoch embeddings
    z_test, y_test = get_fold_test_embeddings_for_bootstrap(sid)

    bootstrap_sims = np.empty(N_BOOT, dtype=float)

    # Bootstrap resampling
    for b in range(N_BOOT):

        boot_centroids = []

        for label in TASK_LABELS:

            z_label = z_test[y_test == label]

            if len(z_label) == 0:
                raise ValueError(
                    f"Subject {sid}: no held-out embeddings "
                    f"for workload label {label}."
                )

            idx = bootstrap_rng.integers(
                0,
                len(z_label),
                size=len(z_label)
            )

            boot_centroid = z_label[idx].mean(axis=0)
            boot_centroids.append(boot_centroid)

        boot_T = np.vstack(boot_centroids)

        bootstrap_sims[b] = 1.0 - procrustes(pop_T, boot_T)[2]

    # Bootstrap percentile interval
    ci_low, ci_high = np.quantile(
        bootstrap_sims,
        [0.025, 0.975]
    )

    bootstrap_rows.append({
        "subject": sid,
        "observed_procrustes": observed_sim,
        "bootstrap_mean": bootstrap_sims.mean(),
        "bootstrap_sd": bootstrap_sims.std(ddof=1),
        "ci_2.5": ci_low,
        "ci_97.5": ci_high,
        "ci_width": ci_high - ci_low,
    })


# ------------------------------------------------------------
# Results
# ------------------------------------------------------------

bootstrap_df = (
    pd.DataFrame(bootstrap_rows)
    .sort_values("subject")
    .reset_index(drop=True)
)

display(bootstrap_df.round(5))

print("\n" + "=" * 70)
print("CORRECTED PROCRUSTES BOOTSTRAP SUMMARY")
print("=" * 70)

print(
    f"Mean authoritative Stage-1 Procrustes: "
    f"{bootstrap_df['observed_procrustes'].mean():.5f}"
)

print(
    f"Mean bootstrap Procrustes: "
    f"{bootstrap_df['bootstrap_mean'].mean():.5f}"
)

print(
    f"Mean bootstrap SD: "
    f"{bootstrap_df['bootstrap_sd'].mean():.5f}"
)

print(
    f"Median 95% CI width: "
    f"{bootstrap_df['ci_width'].median():.5f}"
)


# ------------------------------------------------------------
# Sanity check against authoritative Stage-1 CSV
# ------------------------------------------------------------

authoritative_mean = stage1_df["procrustes_similarity"].mean()
bootstrap_observed_mean = bootstrap_df["observed_procrustes"].mean()
difference = abs(authoritative_mean - bootstrap_observed_mean)

print("\nSanity check:")
print(f"Authoritative Stage-1 CSV mean : {authoritative_mean:.6f}")
print(f"Bootstrap observed mean        : {bootstrap_observed_mean:.6f}")
print(f"Absolute difference            : {difference:.6e}")

if np.isclose(authoritative_mean, bootstrap_observed_mean, atol=1e-6):
    print("PASS: bootstrap observed values match Stage-1 CSV.")
else:
    print("WARNING: bootstrap observed values do NOT match Stage-1 CSV.")

,subject,observed_procrustes,bootstrap_mean,bootstrap_sd,ci_2.5,ci_97.5,ci_width
0,1,0.90342,0.76120,0.16714,0.37010,0.97370,0.60360
1,2,0.96286,0.95845,0.02151,0.91003,0.98992,0.07989
2,3,0.96542,0.92922,0.04806,0.80355,0.98453,0.18098
3,4,0.80954,0.80843,0.06875,0.66305,0.93646,0.27341
4,5,0.91273,0.87951,0.08550,0.67489,0.98172,0.30683
5,6,0.90793,0.88779,0.07003,0.72158,0.98033,0.25875
6,7,0.99841,0.95031,0.05239,0.79567,0.99713,0.20146
7,8,0.96452,0.94076,0.04669,0.82602,0.99315,0.16713
8,9,0.94396,0.93581,0.04022,0.83987,0.99355,0.15369
9,10,0.97413,0.96004,0.03494,0.86599,0.99751,0.13152



CORRECTED PROCRUSTES BOOTSTRAP SUMMARY
Mean authoritative Stage-1 Procrustes: 0.87747
Mean bootstrap Procrustes: 0.85354
Mean bootstrap SD: 0.06253
Median 95% CI width: 0.21771

Sanity check:
Authoritative Stage-1 CSV mean : 0.877468
Bootstrap observed mean        : 0.877468
Absolute difference            : 0.000000e+00
PASS: bootstrap observed values match Stage-1 CSV.


In [20]:
# ============================================================
# 6A. BOOTSTRAP 95% CI BY SUBJECT — PLOTLY
# ============================================================

x = np.arange(len(bootstrap_df))

observed = bootstrap_df["observed_procrustes"].to_numpy()
lower = bootstrap_df["ci_2.5"].to_numpy()
upper = bootstrap_df["ci_97.5"].to_numpy()

fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=x,
        y=observed,
        mode="markers",
        name="Observed",
        error_y=dict(
            type="data",
            symmetric=False,
            array=upper - observed,
            arrayminus=observed - lower,
            thickness=1.2,
            width=4
        ),
        hovertemplate=(
            "Subject %{customdata[0]}<br>"
            "Observed: %{y:.3f}<br>"
            "95% CI: [%{customdata[1]:.3f}, %{customdata[2]:.3f}]"
            "<extra></extra>"
        ),
        customdata=np.column_stack([
            bootstrap_df["subject"].to_numpy(),
            lower,
            upper
        ])
    )
)

fig.update_layout(
    title="Bootstrap Stability of Procrustes Similarity",
    xaxis=dict(
        title="Held-out subject",
        tickmode="array",
        tickvals=x,
        ticktext=bootstrap_df["subject"].astype(int)
    ),
    yaxis=dict(
        title="Procrustes similarity"
    ),
    template="plotly_white",
    height=500,
    width=1100,
    hovermode="closest"
)

fig.show()

## 7. REST-anchor validation

This is a Stage-2-supporting analysis.

**Question:**

> Does the frozen encoder place REST in a geometrically distinct region relative to the N-back states, making it a plausible subject-specific reference point?

REST, 0-back, 1-back, and 2-back centroids are computed separately for each held-out subject using the frozen fold-specific LOSO encoder. REST-to-N-back distances are evaluated in the original 64-D latent space.

A representative held-out subject is additionally visualized using a joint PCA projection of **REST + 0-back + 1-back + 2-back**.

The four-class silhouette is retained only as a **descriptive cluster diagnostic**.

In [10]:
# ============================================================
# 7. REST ANCHOR — ALL 29 SUBJECTS
# ============================================================

rest_rows = []

for sid in SUBJECTS:
    graphs = load_graphs_once()

    train_g = [
        g for g in graphs
        if get_subject(g) != sid and get_label(g) in TASK_LABELS
    ]
    test_g = [
        g for g in graphs
        if get_subject(g) == sid
    ]

    model = load_fold_model(sid, "fignn_best.pth")
    mean, std = load_fold_stats(sid, train_g)

    # Current helper expects (model, graphs, norm_stats)
    z, y = encode_graphs(
        model,
        test_g,
        mean,
        std
    )

    if not all(np.any(y == l) for l in [REST_LABEL, 0, 1, 2]):
        print(f"Subject {sid}: missing REST/0/1/2; skipped.")
        continue

    cR = centroid(z, y, REST_LABEL)
    c0 = centroid(z, y, 0)
    c1 = centroid(z, y, 1)
    c2 = centroid(z, y, 2)

    dR0 = np.linalg.norm(cR - c0)
    dR1 = np.linalg.norm(cR - c1)
    dR2 = np.linalg.norm(cR - c2)

    rest_rows.append({
        "subject": sid,
        "d_REST_0B": dR0,
        "d_REST_1B": dR1,
        "d_REST_2B": dR2,
        "d_REST_1B_over_0B": (
            dR1 / dR0 if dR0 > 1e-12 else np.nan
        ),
        "d_REST_2B_over_0B": (
            dR2 / dR0 if dR0 > 1e-12 else np.nan
        ),
    })

    del model, z

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

rest_df = pd.DataFrame(rest_rows)

display(rest_df.round(5))

print("\nMean REST distances:")
display(
    rest_df[
        ["d_REST_0B", "d_REST_1B", "d_REST_2B"]
    ]
    .agg(["mean", "median", "std"])
    .round(5)
)

,subject,d_REST_0B,d_REST_1B,d_REST_2B,d_REST_1B_over_0B,d_REST_2B_over_0B
0,1,0.16862,0.15956,0.14998,0.94630,0.88948
1,2,0.09805,0.11077,0.18397,1.12967,1.87627
2,3,0.14655,0.15637,0.18443,1.06700,1.25843
3,4,0.15690,0.19581,0.21960,1.24800,1.39960
4,5,0.22957,0.25732,0.28281,1.12091,1.23195
5,6,0.07274,0.09916,0.12941,1.36307,1.77902
6,7,0.17823,0.18853,0.20889,1.05780,1.17203
7,8,0.10740,0.12092,0.14333,1.12584,1.33455
8,9,0.22128,0.21975,0.28253,0.99310,1.27682
9,10,0.06057,0.06664,0.12474,1.10026,2.05951



Mean REST distances:


,d_REST_0B,d_REST_1B,d_REST_2B
mean,0.11364,0.12450,0.15606
median,0.10290,0.11077,0.14998
std,0.04929,0.05370,0.05651


In [11]:
# ============================================================
# 7A. REST AND N-BACK LATENT DISTANCES — PLOTLY
# ============================================================

plot_df = pd.DataFrame({
    "State": (
        ["REST"] * len(rest_df)
        + ["0-back"] * len(rest_df)
        + ["1-back"] * len(rest_df)
        + ["2-back"] * len(rest_df)
    ),
    "Distance": (
        [0.0] * len(rest_df)
        + rest_df["d_REST_0B"].tolist()
        + rest_df["d_REST_1B"].tolist()
        + rest_df["d_REST_2B"].tolist()
    ),
    "Subject": (
        rest_df["subject"].tolist() * 4
    )
})

fig = go.Figure()

states = ["REST", "0-back", "1-back", "2-back"]

for state in states:
    subset = plot_df[plot_df["State"] == state]

    fig.add_trace(
        go.Scatter(
            x=subset["State"],
            y=subset["Distance"],
            mode="markers",
            name=state,
            customdata=subset["Subject"],
            hovertemplate=(
                "State: %{x}<br>"
                "Subject: %{customdata}<br>"
                "Distance from REST: %{y:.3f}"
                "<extra></extra>"
            )
        )
    )

# Mean markers
means = [
    0.0,
    rest_df["d_REST_0B"].mean(),
    rest_df["d_REST_1B"].mean(),
    rest_df["d_REST_2B"].mean()
]

fig.add_trace(
    go.Scatter(
        x=states,
        y=means,
        mode="markers",
        name="Mean",
        marker=dict(
            size=12,
            symbol="line-ew",
            line=dict(width=3)
        ),
        hovertemplate=(
            "State: %{x}<br>"
            "Mean distance: %{y:.3f}"
            "<extra></extra>"
        )
    )
)

fig.update_layout(
    title="Latent Position of REST and N-back States",
    xaxis=dict(
        title="State",
        categoryorder="array",
        categoryarray=states
    ),
    yaxis=dict(
        title="Distance from REST in 64-D latent space"
    ),
    template="plotly_white",
    height=500,
    width=900,
    hovermode="closest"
)

fig.show()
fig.write_image("rest_nback_latent_distances.pdf")

In [12]:
# ============================================================
# 7B. EXISTING GEOMETRY — CONSOLIDATED
# ============================================================

geometry_cols=["subject","d01","d12","d02","d01_d02","d12_d02","angle_deg","path_efficiency","centrality","oneback_between"]
display(stage1_df[geometry_cols].sort_values("subject").round(5))

print("\nCurrent Stage-1 summary:")
print(f"Mean d01/d02: {stage1_df['d01_d02'].mean():.4f} ± {stage1_df['d01_d02'].std():.4f}")
print(f"Mean d12/d02: {stage1_df['d12_d02'].mean():.4f} ± {stage1_df['d12_d02'].std():.4f}")
print(f"Mean trajectory angle: {stage1_df['angle_deg'].mean():.2f} ± {stage1_df['angle_deg'].std():.2f}°")
print(f"Mean path efficiency: {stage1_df['path_efficiency'].mean():.4f} ± {stage1_df['path_efficiency'].std():.4f}")
print(f"Mean centrality: {stage1_df['centrality'].mean():.4f} ± {stage1_df['centrality'].std():.4f}")

,subject,d01,d12,d02,d01_d02,d12_d02,angle_deg,path_efficiency,centrality,oneback_between
0,1,0.01212,0.01493,0.02168,0.55920,0.68878,73.95782,0.80129,0.83828,True
1,2,0.02468,0.07759,0.08846,0.27902,0.87715,71.81416,0.86493,0.30846,True
2,3,0.01733,0.03357,0.04422,0.39203,0.75914,63.02120,0.86868,0.57739,True
3,4,0.04344,0.02820,0.06735,0.64496,0.41865,40.80778,0.94019,0.75929,True
4,5,0.03219,0.03390,0.05993,0.53717,0.56562,49.88506,0.90680,0.96863,True
5,6,0.03246,0.03569,0.05967,0.54395,0.59801,57.81634,0.87569,0.93827,True
6,7,0.01183,0.02573,0.03604,0.32820,0.71385,35.24010,0.95965,0.59813,True
7,8,0.02024,0.02668,0.04549,0.44500,0.58653,28.68128,0.96944,0.85401,True
8,9,0.01081,0.06517,0.06446,0.16778,1.01102,98.52725,0.84832,0.00599,True
9,10,0.01291,0.06195,0.06950,0.18573,0.89132,58.89511,0.92846,0.24004,True



Current Stage-1 summary:
Mean d01/d02: 0.4772 ± 0.2695
Mean d12/d02: 0.8239 ± 0.2839
Mean trajectory angle: 73.20 ± 39.67°
Mean path efficiency: 0.8192 ± 0.1704
Mean centrality: 0.4214 ± 0.4572


## 8. Stage-2 calibration — descriptive summary

The Stage-2 calibration experiment is evaluated separately from the
Stage-1 robustness analyses.

The authoritative Stage-2 results are loaded from the dedicated
Stage-2 output files. This notebook reports only the descriptive
64-D estimation errors and does not recompute inferential tests.

In [13]:
stage2 = pd.read_csv(STAGE2_CAL_PATH)

baseline = stage2[stage2["method"] == "population_only"][["subject", "error_64D"]].rename(columns={"error_64D": "population_error"})

methods = stage2[stage2["method"] != "population_only"].copy()

stage2_comparison = methods.merge(baseline, on="subject")

stage2_comparison["error_reduction"] = (
    stage2_comparison["population_error"] - stage2_comparison["error_64D"]
)

stage2_comparison["improved"] = (
    stage2_comparison["error_64D"] < stage2_comparison["population_error"]
)

display(stage2_comparison.groupby("method").agg(
    mean_error=("error_64D", "mean"),
    mean_reduction=("error_reduction", "mean"),
    improved_subjects=("improved", "sum"),
    n=("improved", "count"),
))

,mean_error,mean_reduction,improved_subjects,n
method,,,,
frozen_geometric,0.034542,0.030317,22,29
midpoint,0.052229,0.012630,18,29
regression,0.050849,0.014011,19,29


## 9. Architecture sensitivity ablation

A 29-fold subject-wise architecture-sensitivity experiment compares the
historical 64/4/32 FIGNN architecture with the final 128/8/64 architecture.

Both architectures are evaluated using the same held-out subject in each
LOSO fold. The historical architecture uses its fold-specific checkpoint
and training-derived normalization statistics, while the final architecture
uses the corresponding authoritative LOSO checkpoint and normalization
statistics.

For each architecture and held-out subject, the analysis:

1. loads the saved best checkpoint;
2. applies the corresponding training-derived normalization;
3. encodes the held-out subject;
4. computes the held-out 0-back, 1-back, and 2-back centroids;
5. constructs the training-subject population trajectory;
6. computes normalized trajectory geometry;
7. computes Procrustes similarity between the held-out and population
   trajectories.

The primary cross-dimensional comparisons use dimension-normalized geometry:
\(d_{01}/d_{02}\), \(d_{12}/d_{02}\), trajectory angle, and path efficiency.
Procrustes similarity is reported as a secondary alignment measure.

Raw Euclidean distances are retained for reference but are not directly
compared between the 32-D and 64-D latent spaces.

This is an architecture-sensitivity analysis and does not replace the
authoritative final 29-fold Stage-1 evaluation.

In [14]:
# ============================================================
# 9. ARCHITECTURE ABLATION
#    Historical 64/4/32 vs Final 128/8/64
# ============================================================

import os
import pickle
import numpy as np
import pandas as pd
import torch
from scipy.spatial import procrustes

# ============================================================
# SETTINGS
# ============================================================

ABLATION_ROOT = "/lp-dev/24BCE1793/outputs/ablation_64_4_32"
LOSO_ROOT = "/lp-dev/24BCE1793/outputs/loso"
ABLATION_SUBJECTS = list(range(1, 30))
SMALL_HIDDEN = 64
SMALL_HEADS = 4
SMALL_LATENT = 32
FINAL_HIDDEN = 128
FINAL_HEADS = 8
FINAL_LATENT = 64
DROPOUT = 0.3

# ============================================================
# LOAD DATA
# ============================================================

with open(GRAPH_PATH, "rb") as f:
    all_graphs_ablation = pickle.load(f)

all_graphs_ablation = [g for g in all_graphs_ablation if get_label(g) in TASK_LABELS]

print(f"Graphs available for ablation: {len(all_graphs_ablation):,}")
print(f"Subjects: {len(ABLATION_SUBJECTS)}")
print(f"Historical architecture: {SMALL_HIDDEN}/{SMALL_HEADS}/{SMALL_LATENT}")
print(f"Final architecture: {FINAL_HIDDEN}/{FINAL_HEADS}/{FINAL_LATENT}")

# ============================================================
# LOAD HISTORICAL 64/4/32 MODEL
# ============================================================

def load_small_model(subject):
    ckpt_path = os.path.join(ABLATION_ROOT, f"subject_{subject:02d}", "fignn_best.pth")
    if not os.path.exists(ckpt_path):
        raise FileNotFoundError(ckpt_path)
    model = FIGNNEncoder(hidden_dim=SMALL_HIDDEN, n_heads=SMALL_HEADS, latent_dim=SMALL_LATENT, dropout=DROPOUT).to(DEVICE)
    checkpoint = torch.load(ckpt_path, map_location=DEVICE, weights_only=False)
    if isinstance(checkpoint, dict):
        if "state_dict" in checkpoint:
            state_dict = checkpoint["state_dict"]
        elif "model_state_dict" in checkpoint:
            state_dict = checkpoint["model_state_dict"]
        else:
            state_dict = checkpoint
    else:
        state_dict = checkpoint
    model.load_state_dict(state_dict)
    model.eval()
    return model

# ============================================================
# LOAD HISTORICAL NORMALISATION
# ============================================================

def load_small_stats(subject):
    stats_path = os.path.join(ABLATION_ROOT, f"subject_{subject:02d}", "normalisation_stats.pt")
    if not os.path.exists(stats_path):
        raise FileNotFoundError(stats_path)
    stats = torch.load(stats_path, map_location="cpu", weights_only=False)
    return {"mean": stats["mean"].float(), "std": stats["std"].float().clamp(min=1e-8)}

# ============================================================
# LOAD FINAL 128/8/64 MODEL
# ============================================================

def load_final_model(subject):
    ckpt_path = os.path.join(LOSO_ROOT, f"subject_{subject:02d}", "fignn_best.pth")
    if not os.path.exists(ckpt_path):
        raise FileNotFoundError(ckpt_path)
    model = FIGNNEncoder(hidden_dim=FINAL_HIDDEN, n_heads=FINAL_HEADS, latent_dim=FINAL_LATENT, dropout=DROPOUT).to(DEVICE)
    checkpoint = torch.load(ckpt_path, map_location=DEVICE, weights_only=False)
    if isinstance(checkpoint, dict):
        if "state_dict" in checkpoint:
            state_dict = checkpoint["state_dict"]
        elif "model_state_dict" in checkpoint:
            state_dict = checkpoint["model_state_dict"]
        else:
            state_dict = checkpoint
    else:
        state_dict = checkpoint
    model.load_state_dict(state_dict)
    model.eval()
    return model

# ============================================================
# LOAD FINAL LOSO NORMALISATION
# ============================================================

def load_final_stats(subject):
    stats_path = os.path.join(LOSO_ROOT, f"subject_{subject:02d}", "normalisation_stats.pt")
    if not os.path.exists(stats_path):
        raise FileNotFoundError(stats_path)
    stats = torch.load(stats_path, map_location="cpu", weights_only=False)
    return {"mean": stats["mean"].float(), "std": stats["std"].float().clamp(min=1e-8)}

# ============================================================
# POPULATION TRAJECTORY
# ============================================================

def build_population_trajectory(model, train_graphs, norm_stats):
    training_subjects = sorted(set(get_subject(g) for g in train_graphs))
    subject_centroids = []
    for sid in training_subjects:
        sid_graphs = [g for g in train_graphs if get_subject(g) == sid]
        z_sid, y_sid = encode_graphs(model, sid_graphs, norm_stats["mean"], norm_stats["std"])
        T_sid = trajectory(z_sid, y_sid)
        subject_centroids.append(T_sid)
    return np.mean(np.stack(subject_centroids, axis=0), axis=0)

# ============================================================
# TRAJECTORY METRICS
# ============================================================

def compute_ablation_metrics(T):
    d01, d12, d02, angle, efficiency = trajectory_geometry(T)
    return {"d01": d01, "d12": d12, "d02": d02, "d01_d02": d01 / d02 if d02 > 1e-12 else np.nan, "d12_d02": d12 / d02 if d02 > 1e-12 else np.nan, "angle": angle, "path_efficiency": efficiency}

# ============================================================
# PROCRUSTES
# ============================================================

def compute_procrustes_similarity(population_T, subject_T):
    if population_T.shape != subject_T.shape:
        raise ValueError(f"Trajectory shapes differ: {population_T.shape} vs {subject_T.shape}")
    _, _, disparity = procrustes(population_T, subject_T)
    return 1.0 - disparity

# ============================================================
# PARAMETER COUNTS
# ============================================================

small_reference = FIGNNEncoder(hidden_dim=SMALL_HIDDEN, n_heads=SMALL_HEADS, latent_dim=SMALL_LATENT, dropout=DROPOUT)
final_reference = FIGNNEncoder(hidden_dim=FINAL_HIDDEN, n_heads=FINAL_HEADS, latent_dim=FINAL_LATENT, dropout=DROPOUT)
small_params = sum(p.numel() for p in small_reference.parameters() if p.requires_grad)
final_params = sum(p.numel() for p in final_reference.parameters() if p.requires_grad)
del small_reference
del final_reference

parameter_table = pd.DataFrame({"Architecture": ["Historical", "Final"], "Hidden dimension": [SMALL_HIDDEN, FINAL_HIDDEN], "Attention heads": [SMALL_HEADS, FINAL_HEADS], "Latent dimension": [SMALL_LATENT, FINAL_LATENT], "Trainable parameters": [small_params, final_params]})

display(parameter_table)

# ============================================================
# RUN PAIRED 29-FOLD EVALUATION
# ============================================================

rows = []

for subject in ABLATION_SUBJECTS:
    print(f"\nEvaluating subject {subject:02d}")
    train_graphs = [g for g in all_graphs_ablation if get_subject(g) != subject]
    test_graphs = [g for g in all_graphs_ablation if get_subject(g) == subject]

    # --------------------------------------------------------
    # HISTORICAL 64/4/32
    # --------------------------------------------------------

    small_model = load_small_model(subject)
    small_stats = load_small_stats(subject)
    small_z_test, small_y_test = encode_graphs(small_model, test_graphs, small_stats["mean"], small_stats["std"])
    small_subject_T = trajectory(small_z_test, small_y_test)
    small_population_T = build_population_trajectory(small_model, train_graphs, small_stats)
    small_metrics = compute_ablation_metrics(small_subject_T)
    small_procrustes = compute_procrustes_similarity(small_population_T, small_subject_T)

    # --------------------------------------------------------
    # FINAL 128/8/64
    # --------------------------------------------------------

    final_model = load_final_model(subject)
    final_stats = load_final_stats(subject)
    final_z_test, final_y_test = encode_graphs(final_model, test_graphs, final_stats["mean"], final_stats["std"])
    final_subject_T = trajectory(final_z_test, final_y_test)
    final_population_T = build_population_trajectory(final_model, train_graphs, final_stats)
    final_metrics = compute_ablation_metrics(final_subject_T)
    final_procrustes = compute_procrustes_similarity(final_population_T, final_subject_T)

    rows.append({"subject": subject, "small_d01": small_metrics["d01"], "small_d12": small_metrics["d12"], "small_d02": small_metrics["d02"], "small_d01_d02": small_metrics["d01_d02"], "small_d12_d02": small_metrics["d12_d02"], "small_angle": small_metrics["angle"], "small_path_efficiency": small_metrics["path_efficiency"], "small_procrustes": small_procrustes, "final_d01": final_metrics["d01"], "final_d12": final_metrics["d12"], "final_d02": final_metrics["d02"], "final_d01_d02": final_metrics["d01_d02"], "final_d12_d02": final_metrics["d12_d02"], "final_angle": final_metrics["angle"], "final_path_efficiency": final_metrics["path_efficiency"], "final_procrustes": final_procrustes, "small_d01_lt_d02": small_metrics["d01"] < small_metrics["d02"], "final_d01_lt_d02": final_metrics["d01"] < final_metrics["d02"]})

# ============================================================
# SUBJECT-WISE RESULTS
# ============================================================

ablation_df = pd.DataFrame(rows)

display(ablation_df.round(4))

# ============================================================
# SUMMARY STATISTICS
# ============================================================

n_subjects = len(ablation_df)

summary_metrics = pd.DataFrame({"Metric": ["d01 / d02", "d12 / d02", "Trajectory angle (deg)", "Path efficiency", "Procrustes similarity"], "64/4/32 mean": [ablation_df["small_d01_d02"].mean(), ablation_df["small_d12_d02"].mean(), ablation_df["small_angle"].mean(), ablation_df["small_path_efficiency"].mean(), ablation_df["small_procrustes"].mean()], "64/4/32 SD": [ablation_df["small_d01_d02"].std(), ablation_df["small_d12_d02"].std(), ablation_df["small_angle"].std(), ablation_df["small_path_efficiency"].std(), ablation_df["small_procrustes"].std()], "128/8/64 mean": [ablation_df["final_d01_d02"].mean(), ablation_df["final_d12_d02"].mean(), ablation_df["final_angle"].mean(), ablation_df["final_path_efficiency"].mean(), ablation_df["final_procrustes"].mean()], "128/8/64 SD": [ablation_df["final_d01_d02"].std(), ablation_df["final_d12_d02"].std(), ablation_df["final_angle"].std(), ablation_df["final_path_efficiency"].std(), ablation_df["final_procrustes"].std()]})

display(summary_metrics.round(4))

# ============================================================
# PRIMARY ORDERING
# ============================================================

ordering_table = pd.DataFrame({"Architecture": ["64/4/32", "128/8/64"], "Subjects with d01 < d02": [int(ablation_df["small_d01_lt_d02"].sum()), int(ablation_df["final_d01_lt_d02"].sum())], "Total subjects": [n_subjects, n_subjects]})

ordering_table["Proportion"] = ordering_table["Subjects with d01 < d02"] / ordering_table["Total subjects"]

display(ordering_table.round(4))

# ============================================================
# PAIRED DIFFERENCES
# ============================================================

comparison_df = pd.DataFrame({"Metric": ["d01 / d02", "d12 / d02", "Trajectory angle (deg)", "Path efficiency", "Procrustes similarity"], "Mean 128/8/64 − 64/4/32": [(ablation_df["final_d01_d02"] - ablation_df["small_d01_d02"]).mean(), (ablation_df["final_d12_d02"] - ablation_df["small_d12_d02"]).mean(), (ablation_df["final_angle"] - ablation_df["small_angle"]).mean(), (ablation_df["final_path_efficiency"] - ablation_df["small_path_efficiency"]).mean(), (ablation_df["final_procrustes"] - ablation_df["small_procrustes"]).mean()], "Median 128/8/64 − 64/4/32": [(ablation_df["final_d01_d02"] - ablation_df["small_d01_d02"]).median(), (ablation_df["final_d12_d02"] - ablation_df["small_d12_d02"]).median(), (ablation_df["final_angle"] - ablation_df["small_angle"]).median(), (ablation_df["final_path_efficiency"] - ablation_df["small_path_efficiency"]).median(), (ablation_df["final_procrustes"] - ablation_df["small_procrustes"]).median()]})

display(comparison_df.round(4))

# ============================================================
# SAVE RESULTS
# ============================================================

ABLATION_RESULTS_PATH = os.path.join(ABLATION_ROOT, "architecture_ablation_29fold_subject_results.csv")
ABLATION_SUMMARY_PATH = os.path.join(ABLATION_ROOT, "architecture_ablation_29fold_summary.csv")
ABLATION_ORDERING_PATH = os.path.join(ABLATION_ROOT, "architecture_ablation_29fold_ordering.csv")
ABLATION_COMPARISON_PATH = os.path.join(ABLATION_ROOT, "architecture_ablation_29fold_paired_differences.csv")

ablation_df.to_csv(ABLATION_RESULTS_PATH, index=False)
summary_metrics.to_csv(ABLATION_SUMMARY_PATH, index=False)
ordering_table.to_csv(ABLATION_ORDERING_PATH, index=False)
comparison_df.to_csv(ABLATION_COMPARISON_PATH, index=False)

print(f"\nSaved subject-wise results: {ABLATION_RESULTS_PATH}")
print(f"Saved summary: {ABLATION_SUMMARY_PATH}")
print(f"Saved ordering results: {ABLATION_ORDERING_PATH}")
print(f"Saved paired differences: {ABLATION_COMPARISON_PATH}")

Graphs available for ablation: 25,056
Subjects: 29
Historical architecture: 64/4/32
Final architecture: 128/8/64


,Architecture,Hidden dimension,Attention heads,Latent dimension,Trainable parameters
0,Historical,64,4,32,12100
1,Final,128,8,64,80516



Evaluating subject 01

Evaluating subject 02

Evaluating subject 03

Evaluating subject 04

Evaluating subject 05

Evaluating subject 06

Evaluating subject 07

Evaluating subject 08

Evaluating subject 09

Evaluating subject 10

Evaluating subject 11

Evaluating subject 12

Evaluating subject 13

Evaluating subject 14

Evaluating subject 15

Evaluating subject 16

Evaluating subject 17

Evaluating subject 18

Evaluating subject 19

Evaluating subject 20

Evaluating subject 21

Evaluating subject 22

Evaluating subject 23

Evaluating subject 24

Evaluating subject 25

Evaluating subject 26

Evaluating subject 27

Evaluating subject 28

Evaluating subject 29


,subject,small_d01,small_d12,small_d02,small_d01_d02,small_d12_d02,small_angle,small_path_efficiency,small_procrustes,final_d01,final_d12,final_d02,final_d01_d02,final_d12_d02,final_angle,final_path_efficiency,final_procrustes,small_d01_lt_d02,final_d01_lt_d02
0,1,0.0109,0.0112,0.0178,0.6115,0.6305,107.236198,0.8051,0.8743,0.0121,0.0149,0.0217,0.5592,0.6888,106.042297,0.8013,0.9034,True,True
1,2,0.0269,0.0696,0.0509,0.5287,1.3670,37.398800,0.5275,0.7964,0.0247,0.0776,0.0885,0.2790,0.8771,108.185799,0.8649,0.9629,True,True
2,3,0.0134,0.0333,0.0291,0.4596,1.1422,60.570599,0.6243,0.8614,0.0173,0.0336,0.0442,0.3920,0.7591,116.978600,0.8687,0.9654,True,True
3,4,0.0508,0.0240,0.0729,0.6976,0.3297,151.612793,0.9734,0.7115,0.0434,0.0282,0.0674,0.6450,0.4187,139.192200,0.9402,0.8095,True,True
4,5,0.0110,0.0191,0.0242,0.4536,0.7896,103.788696,0.8044,0.9808,0.0322,0.0339,0.0599,0.5372,0.5656,130.114899,0.9068,0.9127,True,True
5,6,0.0263,0.0288,0.0510,0.5150,0.5646,135.678802,0.9263,0.9032,0.0325,0.0357,0.0597,0.5439,0.5980,122.183800,0.8757,0.9079,True,True
6,7,0.0059,0.0340,0.0354,0.1672,0.9615,98.499901,0.8859,0.9618,0.0118,0.0257,0.0360,0.3282,0.7139,144.759705,0.9596,0.9984,True,True
7,8,0.0190,0.0242,0.0399,0.4754,0.6055,135.036102,0.9251,0.9032,0.0202,0.0267,0.0455,0.4450,0.5865,151.318695,0.9694,0.9645,True,True
8,9,0.0113,0.0514,0.0587,0.1929,0.8755,125.528397,0.9360,0.9844,0.0108,0.0652,0.0645,0.1678,1.0110,81.472801,0.8483,0.9440,True,True
9,10,0.0130,0.0511,0.0595,0.2191,0.8589,124.699898,0.9276,0.9967,0.0129,0.0619,0.0695,0.1857,0.8913,121.104698,0.9285,0.9741,True,True


,Metric,64/4/32 mean,64/4/32 SD,128/8/64 mean,128/8/64 SD
0,d01 / d02,0.5865,0.4992,0.4772,0.2695
1,d12 / d02,0.9240,0.6056,0.8239,0.2839
2,Trajectory angle (deg),99.0501,39.1663,106.8030,39.6714
3,Path efficiency,0.7706,0.1996,0.8192,0.1704
4,Procrustes similarity,0.8556,0.1566,0.8775,0.1548


,Architecture,Subjects with d01 < d02,Total subjects,Proportion
0,64/4/32,26,29,0.8966
1,128/8/64,27,29,0.9310


,Metric,Mean 128/8/64 − 64/4/32,Median 128/8/64 − 64/4/32
0,d01 / d02,-0.1094,-0.0251
1,d12 / d02,-0.1001,-0.0358
2,Trajectory angle (deg),7.7529,4.8510
3,Path efficiency,0.0485,0.0179
4,Procrustes similarity,0.0219,0.0047



Saved subject-wise results: /lp-dev/24BCE1793/outputs/ablation_64_4_32/architecture_ablation_29fold_subject_results.csv
Saved summary: /lp-dev/24BCE1793/outputs/ablation_64_4_32/architecture_ablation_29fold_summary.csv
Saved ordering results: /lp-dev/24BCE1793/outputs/ablation_64_4_32/architecture_ablation_29fold_ordering.csv
Saved paired differences: /lp-dev/24BCE1793/outputs/ablation_64_4_32/architecture_ablation_29fold_paired_differences.csv


### Formal ablation decision

A controlled 29-fold architecture sensitivity analysis was performed comparing:

| Configuration | hidden | heads | latent |
|---|---:|---:|---:|
| Smaller | 64 | 4 | 32 |
| Final | 128 | 8 | 64 |

Both configurations are evaluated under subject-wise LOSO using fold-specific training normalization and matched trajectory metrics.

The comparison is treated as an architecture sensitivity analysis rather than evidence of statistical superiority of either configuration.

In [15]:
# ============================================================
# 10. FINAL RESEARCH SUMMARY
# ============================================================

print("=" * 78)
print("FIGNN RESEARCH VALIDATION SUMMARY")
print("=" * 78)

print("\nPRIMARY STAGE 1")

primary_count = int(stage1_df["oneback_between"].sum())
supporting_count = int((stage1_df["d01_d02"] < 1.0).sum())

print(
    f"  1B between 0B and 2B: "
    f"{primary_count}/29 "
    f"({100 * primary_count / 29:.2f}%)"
)

print(
    f"  Supporting d01/d02 < 1: "
    f"{supporting_count}/29 "
    f"({100 * supporting_count / 29:.2f}%)"
)

print(
    f"  Relative 1B position: "
    f"{stage1_df['relative_1back_position'].mean():.4f} ± "
    f"{stage1_df['relative_1back_position'].std(ddof=1):.4f}"
)

print(
    f"  Perpendicular distance: "
    f"{stage1_df['perpendicular_distance'].mean():.4f} ± "
    f"{stage1_df['perpendicular_distance'].std(ddof=1):.4f}"
)

print(
    f"  Path efficiency: "
    f"{stage1_df['path_efficiency'].mean():.4f} ± "
    f"{stage1_df['path_efficiency'].std(ddof=1):.4f}"
)

print(
    f"  Procrustes similarity: "
    f"{stage1_df['procrustes_similarity'].mean():.4f} ± "
    f"{stage1_df['procrustes_similarity'].std(ddof=1):.4f}"
)

print("\nPROCRUSTES ROBUSTNESS")

if "perm_df" in globals():
    print(
        f"  Permutations per subject: {N_PERM}"
    )
    print(
        f"  Subjects p < .05: "
        f"{(perm_df['empirical_p_plus1'] < 0.05).sum()}/29"
    )
    print(
        f"  Subjects FDR q < .05: "
        f"{(perm_df['q_fdr'] < 0.05).sum()}/29"
    )

if "bootstrap_df" in globals():
    print(
        f"  Bootstrap subjects: "
        f"{len(bootstrap_df)}/29"
    )

print("\nREST ANCHOR")

if "rest_df" in globals() and len(rest_df):
    print(
        f"  REST → 0B: "
        f"{rest_df['d_REST_0B'].mean():.5f}"
    )
    print(
        f"  REST → 1B: "
        f"{rest_df['d_REST_1B'].mean():.5f}"
    )
    print(
        f"  REST → 2B: "
        f"{rest_df['d_REST_2B'].mean():.5f}"
    )

print("\nSTAGE 2")

stage2_summary = (
    stage2.groupby("method")["error_64D"]
    .agg(["mean", "std", "median", "count"])
    .reindex([
        "population_only",
        "midpoint",
        "regression",
        "frozen_geometric",
    ])
)

display(stage2_summary.round(6))

print("\nARCHITECTURE SENSITIVITY")

if "ablation_df" in globals():
    print(
        f"  Evaluated rows: {len(ablation_df)}"
    )

print("\nValidation notebook completed.")

FIGNN RESEARCH VALIDATION SUMMARY

PRIMARY STAGE 1
  1B between 0B and 2B: 23/29 (79.31%)
  Supporting d01/d02 < 1: 27/29 (93.10%)
  Relative 1B position: 0.2705 ± 0.2906
  Perpendicular distance: 0.0135 ± 0.0088
  Path efficiency: 0.8192 ± 0.1704
  Procrustes similarity: 0.8775 ± 0.1548

PROCRUSTES ROBUSTNESS
  Permutations per subject: 1000
  Subjects p < .05: 0/29
  Subjects FDR q < .05: 0/29
  Bootstrap subjects: 29/29

REST ANCHOR
  REST → 0B: 0.11364
  REST → 1B: 0.12450
  REST → 2B: 0.15606

STAGE 2


,mean,std,median,count
method,,,,
population_only,0.064860,0.031883,0.056985,29
midpoint,0.052229,0.025477,0.050302,29
regression,0.050849,0.030088,0.047213,29
frozen_geometric,0.034542,0.016811,0.029759,29



ARCHITECTURE SENSITIVITY
  Evaluated rows: 29

Validation notebook completed.
